# QA 04 — motion and temporal semantics

Playback controls, decimation, smoothing, fluid playback, complete dashboard composition, and static accessibility modes.

> Run this notebook from top to bottom after installing the project with
> `pip install -e ".[notebooks]"`. Figures are genuine public-API outputs.
> Cells intentionally contain no assertions: automated invariants live in
> `tests/`, while this notebook is for human visual inspection.

In [ ]:
from pathlib import Path
import sys

candidate = Path.cwd().resolve()
while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
if not (candidate / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from inside the Mlektic repository.")
ROOT = candidate
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import HTML, display
from mlektic import export_figure, show_optimized, visualize_logistic, visualize_lr, visualize_nn
from notebooks._support import case_heading, binary_case, linear_case, multiclass_case, torch_xor_case
linear = linear_case(1,estimator='sgd')
logistic = binary_case(1,estimator='sgd')

`steps=K` creates semantic checkpoints. `max_frames=N` retains at most N of
those checkpoints. `interpolation_frames=q` inserts perceptual intervals in
hybrid linear motion; they are not optimizer updates. `frame_duration` controls
playback time, never whether the model trace exists. The dashboard retains the
complete composition and its temporal controls.

### `MOTION-NATIVE-SLOW`

**Inspect:** slow native playback with 8 displayed checkpoints

In [ ]:
case_heading("MOTION-NATIVE-SLOW", "slow native playback with 8 displayed checkpoints")
display(visualize_lr(linear.model,linear.X,linear.y,steps=24,max_frames=8,animation_mode='native',frame_duration=500))

### `MOTION-HYBRID-FLUID`

**Inspect:** fluid hybrid subframes while labels advance only at checkpoints

In [ ]:
case_heading("MOTION-HYBRID-FLUID", "fluid hybrid subframes while labels advance only at checkpoints")
display(visualize_lr(linear.model,linear.X,linear.y,steps=24,max_frames=8,animation_mode='hybrid',interpolation_frames=4,fps=35))

### `MOTION-FRAME-STEP`

**Inspect:** source stride when max_frames is disabled

In [ ]:
case_heading("MOTION-FRAME-STEP", "source stride when max_frames is disabled")
display(visualize_lr(linear.model,linear.X,linear.y,steps=25,max_frames=None,frame_step=4,animation_mode='native'))

### `MOTION-NO-SMOOTH`

**Inspect:** raw replay loss without EMA

In [ ]:
case_heading("MOTION-NO-SMOOTH", "raw replay loss without EMA")
display(visualize_lr(linear.model,linear.X,linear.y,steps=20,max_frames=8,smooth=None,animation_mode='native'))

### `MOTION-LOGISTIC`

**Inspect:** logistic transition duration and redraw-safe model visibility

In [ ]:
case_heading("MOTION-LOGISTIC", "logistic transition duration and redraw-safe model visibility")
display(visualize_logistic(logistic.model,logistic.X,logistic.y,steps=18,max_frames=7,show_loss=True,frame_duration=420,transition_duration=280))

### `MOTION-REDUCED`

**Inspect:** exact final state without playback controls

In [ ]:
case_heading("MOTION-REDUCED", "exact final state without playback controls")
display(visualize_lr(linear.model,linear.X,linear.y,steps=18,max_frames=7,reduced_motion=True,detail='academic'))

### `MOTION-REPORT`

**Inspect:** static report composition as an explicit non-animated format

In [ ]:
case_heading("MOTION-REPORT", "static report composition as an explicit non-animated format")
display(visualize_logistic(logistic.model,logistic.X,logistic.y,steps=18,max_frames=7,format='report',show_loss=True))

## Fluid playback

These figures use the settings that give the smoothest playback: `steps=60`,
`max_frames=60`, and `frame_duration=80` (the library defaults; the transition
is derived from the duration), plus hybrid interpolation at `fps=35` where it
exists (1D linear). Drag the slider and press Play. Steps should follow the
pointer, Play should keep its configured pace, and every equation must look
exactly like the Plotly-playback cell, only faster. The first visit to a step
still redraws the figure (equations are typeset ahead while the figure is
idle); every later visit replays it.

`show_optimized` is used on purpose: JupyterLab and VS Code display plain
figures through Plotly's mimetype extension, which keeps Plotly's own playback.
Colab, classic Jupyter, `show_optimized`, and exported HTML run the fluid
controller. The exported files opened in Chrome are the reference check.

In [ ]:
fluid_linear = visualize_lr(linear.model,linear.X,linear.y,steps=60,max_frames=60,animation_mode='hybrid',interpolation_frames=3,fps=35)
multiclass = multiclass_case(1)
fluid_multiclass = visualize_logistic(multiclass.model,multiclass.X,multiclass.y,steps=60,max_frames=60,frame_duration=80)
multiclass_5d = multiclass_case(5)
fluid_multiclass_5d = visualize_logistic(multiclass_5d.model,multiclass_5d.X,multiclass_5d.y,steps=60,max_frames=60,frame_duration=80)
multiclass_3d = multiclass_case(2)
fluid_multiclass_3d = visualize_logistic(multiclass_3d.model,multiclass_3d.X,multiclass_3d.y,steps=60,max_frames=60,frame_duration=80)

### `MOTION-FLUID-LINEAR-HYBRID`

**Inspect:** hybrid 1D linear playback at 35 fps with fluid stepping

In [ ]:
case_heading("MOTION-FLUID-LINEAR-HYBRID", "hybrid 1D linear playback at 35 fps with fluid stepping")
display(show_optimized(fluid_linear))

### `MOTION-FLUID-LOGISTIC-MULTICLASS`

**Inspect:** multiclass equations change on every step without re-typesetting delay

In [ ]:
case_heading("MOTION-FLUID-LOGISTIC-MULTICLASS", "multiclass equations change on every step without re-typesetting delay")
display(show_optimized(fluid_multiclass))

### `MOTION-FLUID-LOGISTIC-MULTICLASS-PLOTLY`

**Inspect:** the same figure with Plotly playback: identical picture, slower steps

In [ ]:
case_heading("MOTION-FLUID-LOGISTIC-MULTICLASS-PLOTLY", "the same figure with Plotly playback: identical picture, slower steps")
display(show_optimized(fluid_multiclass,fluid_playback=False))

### `MOTION-FLUID-LOGISTIC-5D`

**Inspect:** five-feature multiclass equations stepping fluidly

In [ ]:
case_heading("MOTION-FLUID-LOGISTIC-5D", "five-feature multiclass equations stepping fluidly")
display(show_optimized(fluid_multiclass_5d))

### `MOTION-FLUID-LOGISTIC-3D`

**Inspect:** 3D surface steps: equations are instant, the WebGL surface still rebuilds

In [ ]:
case_heading("MOTION-FLUID-LOGISTIC-3D", "3D surface steps: equations are instant, the WebGL surface still rebuilds")
display(show_optimized(fluid_multiclass_3d))

### `MOTION-FLUID-NN-ACTIVATIONS`

**Inspect:** neural activation equations stepping fluidly across 20 checkpoints

In [ ]:
case_heading("MOTION-FLUID-NN-ACTIVATIONS", "neural activation equations stepping fluidly across 20 checkpoints")
xor_model,xor_X,xor_history = torch_xor_case(steps=40)
display(show_optimized(visualize_nn(xor_model,xor_X[:1],history=xor_history,view='activations',max_frames=20)))

### `MOTION-FLUID-EXPORT`

**Inspect:** exported HTML pairs to open in Chrome: fluid controller versus Plotly playback

In [ ]:
case_heading("MOTION-FLUID-EXPORT", "exported HTML pairs to open in Chrome: fluid controller versus Plotly playback")
out = ROOT/'build'/'qa-fluid'
exported = [
    export_figure(fluid_multiclass,out/'multiclass-fluid.html'),
    export_figure(fluid_multiclass,out/'multiclass-plotly.html',fluid_playback=False),
    export_figure(fluid_multiclass_3d,out/'multiclass-3d-fluid.html'),
    export_figure(fluid_multiclass_3d,out/'multiclass-3d-plotly.html',fluid_playback=False),
]
for path in exported:
    print(path)
links = ''.join(f'<li><a href="{path.as_uri()}" target="_blank">{path.name}</a></li>' for path in exported)
display(HTML(f'<ul>{links}</ul>'))

## Canvas-size coverage

Run the same representative figure in every supported size. Inspect clipping,
mathematical spacing, controls, legends, axes, and any below-plot derivation.

### `MOTION-SIZE-DEFAULT`

**Inspect:** default canvas preset for this notebook's representative figure

In [ ]:
case_heading("MOTION-SIZE-DEFAULT", "default canvas preset for this notebook's representative figure")
display(visualize_lr(linear.model,linear.X,linear.y,steps=12,max_frames=6,animation_mode='native',size='default'))

### `MOTION-SIZE-COMPACT`

**Inspect:** compact canvas preset for this notebook's representative figure

In [ ]:
case_heading("MOTION-SIZE-COMPACT", "compact canvas preset for this notebook's representative figure")
display(visualize_lr(linear.model,linear.X,linear.y,steps=12,max_frames=6,animation_mode='native',size='compact'))

### `MOTION-SIZE-NOTEBOOK`

**Inspect:** notebook canvas preset for this notebook's representative figure

In [ ]:
case_heading("MOTION-SIZE-NOTEBOOK", "notebook canvas preset for this notebook's representative figure")
display(visualize_lr(linear.model,linear.X,linear.y,steps=12,max_frames=6,animation_mode='native',size='notebook'))

### `MOTION-SIZE-WIDE`

**Inspect:** wide canvas preset for this notebook's representative figure

In [ ]:
case_heading("MOTION-SIZE-WIDE", "wide canvas preset for this notebook's representative figure")
display(visualize_lr(linear.model,linear.X,linear.y,steps=12,max_frames=6,animation_mode='native',size='wide'))

### `MOTION-SIZE-CLASSROOM`

**Inspect:** classroom canvas preset for this notebook's representative figure

In [ ]:
case_heading("MOTION-SIZE-CLASSROOM", "classroom canvas preset for this notebook's representative figure")
display(visualize_lr(linear.model,linear.X,linear.y,steps=12,max_frames=6,animation_mode='native',size='classroom'))